# P3_SRR_H4_H8_Fusion_WDice

**Người thực hiện: PhuongTU**

Clone `FUAVX/RPD_Reproduce` và checkout commit Phase 3 đã khóa trong SOURCE_COMMIT. Chỉ cần đính kèm dataset PhenoBench; không cần dataset ZIP source. Hai phương án chạy trong hai session độc lập, mỗi session mặc định 2 GPU, batch 4/GPU, FP32, 100 epochs. Giữ cùng điều kiện cho B1 + WDice đối chứng. Fresh mặc định; resume chỉ khi chỉ định checkpoint, config và manifest của cùng run.


## 1. Dependencies
Bật GPU và Internet cho cell cài dependencies. Nếu Kaggle yêu cầu restart kernel sau pip, restart rồi chạy lại các cell theo thứ tự.

In [ ]:
import os
import signal
import subprocess
import sys

def run(command, cwd=None, env=None, log_path=None):
    print("+", " ".join(map(str, command)), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, env=env, check=True)
        return
    with open(log_path, "a", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1,
                                   start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            # Stop the subprocess and any DDP workers before packaging artifacts.
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            except ProcessLookupError:
                process.wait()
            raise
        finally:
            process.stdout.close()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)

# Keep Kaggle's supplied torch/torchvision/CUDA. Install training dependencies.
run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
     "pytorch-lightning==1.9.5", "torchmetrics==0.11.4", "timm==0.9.16",
     "setuptools<70", "numpy<2", "tensorboard", "matplotlib", "pandas", "tabulate",
     "scikit-image", "tifffile", "pyyaml", "rich"])


## 2. Clone source GitHub, chọn dataset và tạo run riêng
Bật Internet. Giữ SCENARIO đúng notebook và SOURCE_COMMIT đã khóa. Chỉ điền ba trường RESUME khi tiếp tục cùng thí nghiệm.


In [ ]:
import hashlib
import json
import pathlib
import re
import uuid
import zipfile
import yaml

SCENARIO = "P3_SRR_H4_H8_Fusion_WDice"
GITHUB_REPO = "https://github.com/FUAVX/RPD_Reproduce.git"
GITHUB_BRANCH = "codex/p3-srr"  # Fetch location; SOURCE_COMMIT pins runtime code.
SOURCE_COMMIT = "9c9446c360ddaaff78db208c9cc3cb9b810a9b77"
DATASET_DIR = ""  # PhenoBench root; auto-detect when empty.
NUM_DEVICES = 2   # Each notebook uses a separate session with two GPUs.
NUM_WORKERS = 2
RESUME_CHECKPOINT = ""  # Fresh by default; explicit checkpoint only.
RESUME_CONFIG = ""      # active_config.yaml from that checkpoint's run.
RESUME_MANIFEST = ""    # run_manifest.json from the same run (source provenance).
RUN_TOKEN = uuid.uuid4().hex[:12]
RUN_DIR = pathlib.Path("/kaggle/working") / SCENARIO / RUN_TOKEN
RUN_DIR.mkdir(parents=True, exist_ok=False)
REPO_DIR = RUN_DIR / "source"
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
    raise ValueError("SOURCE_COMMIT must be the full reviewed Git commit hash.")
run(["git", "clone", "--filter=blob:none", "--no-checkout", "--single-branch",
     "--branch", GITHUB_BRANCH, GITHUB_REPO, str(REPO_DIR)])
# Exclude historical results/checkpoints from checkout.
run(["git", "sparse-checkout", "init", "--cone"], cwd=REPO_DIR)
run(["git", "sparse-checkout", "set", "models", "datasets", "callbacks", "utils",
     "configs", "tools"], cwd=REPO_DIR)
run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO_DIR)

def git_output(*arguments):
    return subprocess.check_output(["git", *arguments], cwd=REPO_DIR, text=True)

def source_state():
    commit = git_output("rev-parse", "HEAD").strip()
    diff = git_output("diff", "--binary", "HEAD", "--")
    status = git_output("status", "--porcelain=v1", "--untracked-files=all")
    if commit != SOURCE_COMMIT or diff or status:
        raise RuntimeError("Source must remain clean at SOURCE_COMMIT; save changes in a new reviewed commit.")
    return commit, diff, status

source_commit, source_diff, source_status = source_state()
(RUN_DIR / "source_diff.patch").write_text(source_diff, encoding="utf-8")
(RUN_DIR / "source_status.txt").write_text(source_status, encoding="utf-8")

def valid_dataset(path):
    return all((path / split / kind).is_dir()
               for split in ("train", "val") for kind in ("images", "semantics")) and (path / "test/images").is_dir()
if DATASET_DIR:
    dataset = pathlib.Path(DATASET_DIR).resolve()
else:
    dataset_roots = sorted({p.parent.parent for p in pathlib.Path("/kaggle/input").rglob("images")
                            if p.parent.name == "train" and valid_dataset(p.parent.parent)})
    if len(dataset_roots) != 1:
        raise RuntimeError("Attach PhenoBench or set DATASET_DIR; expected one train/val root.")
    dataset = dataset_roots[0]
if not valid_dataset(dataset):
    raise RuntimeError("Incomplete PhenoBench root; loader setup also enumerates test/images.")
if not isinstance(NUM_DEVICES, int) or NUM_DEVICES < 1:
    raise ValueError("NUM_DEVICES must be a positive GPU count.")

source_config = REPO_DIR / "configs/p3_scenarios" / f"{SCENARIO}.yaml"
cfg = yaml.safe_load(source_config.read_text())
cfg["data"]["path_to_dataset"] = str(dataset)
cfg["data"]["num_workers"] = NUM_WORKERS
cfg["train"]["devices"] = NUM_DEVICES
cfg["val"]["devices"] = 1
active_config = RUN_DIR / "active_config.yaml"
active_config.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

resume_values = [RESUME_CHECKPOINT, RESUME_CONFIG, RESUME_MANIFEST]
if any(resume_values) and not all(resume_values):
    raise ValueError("Resume requires checkpoint, its active config and run manifest.")
if RESUME_CHECKPOINT:
    previous_cfg = yaml.safe_load(pathlib.Path(RESUME_CONFIG).read_text())
    for key in ("experiment", "backbone", "train", "loss", "val", "test", "seed"):
        if previous_cfg[key] != cfg[key]:
            raise ValueError(f"Resume config mismatch in {key}; do not cross-resume scenarios/recipes.")
    previous_run = json.loads(pathlib.Path(RESUME_MANIFEST).read_text())
    if previous_run["source_commit"] != source_commit or previous_run["scenario"] != SCENARIO:
        raise ValueError("Resume source/scenario mismatch.")
    if previous_run["config_sha256"] != hashlib.sha256(pathlib.Path(RESUME_CONFIG).read_bytes()).hexdigest():
        raise ValueError("Resume config does not match its recorded manifest.")
    if not pathlib.Path(RESUME_CHECKPOINT).is_file():
        raise FileNotFoundError(RESUME_CHECKPOINT)

MANIFEST = dict(scenario=SCENARIO, run_token=RUN_TOKEN,
                source_repository=GITHUB_REPO, source_branch=GITHUB_BRANCH,
                requested_commit=SOURCE_COMMIT, source_commit=source_commit,
                source_diff=source_diff, source_status=source_status,
                source_config_sha256=hashlib.sha256(source_config.read_bytes()).hexdigest(),
                config_sha256=hashlib.sha256(active_config.read_bytes()).hexdigest(),
                active_config=cfg, dataset_root=str(dataset),
                num_devices=NUM_DEVICES, per_device_batch=cfg["train"]["batch_size"],
                effective_batch=NUM_DEVICES * cfg["train"]["batch_size"] * cfg["train"].get("accumulate_grad_batches", 1),
                resume_checkpoint=RESUME_CHECKPOINT or None,
                resume_config=RESUME_CONFIG or None, resume_manifest=RESUME_MANIFEST or None,
                status="prepared")
(RUN_DIR / "run_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
print("Source commit:", source_commit)
print("Artifacts:", RUN_DIR)
print(yaml.safe_dump({"backbone": cfg["backbone"], "loss": cfg["loss"],
                      "train": cfg["train"]}, sort_keys=False))


## 3. Kiểm tra trước train
Kiểm tra core, import bình thường, batch train 768 và validation theo kích thước mẫu dữ liệu thực tế trên một GPU. Source gốc trả ảnh validation native, không áp dụng center_crop khai báo trong YAML. Đây chưa phải kiểm tra cả dataset hoặc DDP. Nếu OOM, dừng để điều chỉnh đồng nhất recipe.

In [ ]:
# Fresh subprocesses keep checks separate from training RNG and BN state.
import torch
MANIFEST.pop("preflight_passed", None)
MANIFEST["status"] = "checking_preflight"
(RUN_DIR / "run_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
source_state()
if hashlib.sha256(active_config.read_bytes()).hexdigest() != MANIFEST["config_sha256"]:
    raise RuntimeError("Active config changed; prepare a new run and repeat preflight.")
if not torch.cuda.is_available() or torch.cuda.device_count() < NUM_DEVICES:
    raise RuntimeError(f"Need {NUM_DEVICES} visible GPU(s); select the appropriate accelerator.")
MANIFEST["runtime_gpu"] = [dict(name=torch.cuda.get_device_name(i),
                 total_memory=torch.cuda.get_device_properties(i).total_memory)
            for i in range(torch.cuda.device_count())]
MANIFEST["torch"] = torch.__version__
(RUN_DIR / "run_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
run([sys.executable, "-u", "test_p3.py", "--output", str(RUN_DIR / "core_checks.json")],
    cwd=REPO_DIR, log_path=RUN_DIR / "core_checks.log")
run([sys.executable, "-u", "tools/p3_preflight.py", "--config", str(active_config),
     "--output", str(RUN_DIR / "gpu_preflight.json")], cwd=REPO_DIR,
    log_path=RUN_DIR / "gpu_preflight.log")
source_state()
MANIFEST["preflight_passed"] = True
MANIFEST["status"] = "preflight_passed"
(RUN_DIR / "run_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
print("Checks passed. Review peak memory before starting the training cell.")


## 4. Train
Kiểm tra manifest và GPU preflight trước khi chạy cell này. Chỉ số vùng cỏ nhỏ cần bước đánh giá riêng; training hiện có chỉ số pixel/IoU của repository.

In [ ]:
# Fresh training unless the explicit resume fields above were supplied.
source_state()
if hashlib.sha256(active_config.read_bytes()).hexdigest() != MANIFEST["config_sha256"]:
    raise RuntimeError("Active config changed; prepare a new run and repeat preflight.")
if not MANIFEST.get("preflight_passed") or not (RUN_DIR / "gpu_preflight.json").is_file():
    raise RuntimeError("Run the preflight cell successfully before starting training.")
TRAIN_DIR = RUN_DIR / "training_output"
command = [sys.executable, "-u", "multi_metric_train.py", "--config", str(active_config),
           "--export_dir", str(TRAIN_DIR), "--devices", str(NUM_DEVICES)]
if RESUME_CHECKPOINT:
    command += ["--ckpt_path", RESUME_CHECKPOINT, "--resume"]
environment = os.environ.copy()
environment.update(PYTHONUNBUFFERED="1", RUNNER_CAPTURING_LOG="1")
# Lightning 1.9 loads complete trusted training checkpoints on modern PyTorch.
if RESUME_CHECKPOINT:
    environment["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
MANIFEST["training_command"] = command
MANIFEST["status"] = "training"
(RUN_DIR / "run_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
try:
    run(command, cwd=REPO_DIR, env=environment, log_path=RUN_DIR / "training.log")
    MANIFEST["status"] = "complete"
except BaseException as error:
    MANIFEST["status"] = "failed_or_interrupted"
    MANIFEST["error"] = str(error)
    raise
finally:
    (RUN_DIR / "run_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
    # Do not archive the large source tree; it is fingerprinted in the manifest.
    artifact_path = RUN_DIR.parent / f"{SCENARIO}_{RUN_TOKEN}_artifacts.zip"
    with zipfile.ZipFile(artifact_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in RUN_DIR.rglob("*"):
            relative = path.relative_to(RUN_DIR)
            if path.is_file() and relative.parts[0] not in ("source",):
                archive.write(path, str(relative))
    print("Download artifacts:", artifact_path)


## 5. Đóng gói lại kết quả nếu cần

In [ ]:
# Run this cell again after a recoverable interruption to package saved files.
artifact_path = RUN_DIR.parent / f"{SCENARIO}_{RUN_TOKEN}_artifacts.zip"
with zipfile.ZipFile(artifact_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_DIR.rglob("*"):
        relative = path.relative_to(RUN_DIR)
        if path.is_file() and relative.parts[0] not in ("source",):
            archive.write(path, str(relative))
print(artifact_path)
